# 03 — Prompt Consistency & Scoring Experiments

This notebook explores **cross-cutting consistency techniques** that work
regardless of the generation method:

1. **Prompt anchoring** — fixed base prompt + variable action/setting
2. **Seed locking** — same seed across a batch for shared noise patterns
3. **CLIP similarity scoring** — automated ranking of outputs by consistency
4. **Consistency scorer** — unified multi-signal consistency scoring

**References:**
- CLIP paper: https://arxiv.org/abs/2103.00020
- BLIP paper: https://arxiv.org/abs/2201.12086
- Seed reproducibility: https://huggingface.co/docs/diffusers/using-diffusers/reproducibility

## 0. Setup

In [ ]:
import sys

sys.path.insert(0, '..')

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

OUTPUT_DIR = Path('../assets/outputs/03_consistency')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = 'runwayml/stable-diffusion-v1-5'

In [ ]:
# Load the base pipeline (requires src/image/pipeline.py to be implemented)
from src.image.pipeline import ImageGenerationPipeline

pipe = ImageGenerationPipeline(model_id=BASE_MODEL_ID, device='cuda')
print('Pipeline loaded.')

---
## 1. Prompt anchoring

We define a fixed **anchor** (character description) and vary only the action and setting.

In [ ]:
from src.utils.prompt_helpers import PromptTemplate

template = PromptTemplate(
    anchor='sks character, 1girl, blue eyes, silver hair, anime style',
    template='{anchor}, {action}, {setting}, dramatic lighting',
    negative_prompt='photorealistic, realistic, photograph, 3D render, blurry, deformed, extra limbs',
    quality_tags=['masterpiece', 'best quality', 'highly detailed'],
)

variations = [
    {'action': 'standing on a mountain peak', 'setting': 'golden hour, clear sky'},
    {'action': 'walking through a neon-lit city', 'setting': 'night, rain-slicked streets'},
    {'action': 'sitting by a campfire', 'setting': 'forest, starry night'},
    {'action': 'reading a book', 'setting': 'cosy library, warm lamp light'},
]

prompts = template.build_batch(variations)
for i, p in enumerate(prompts):
    print(f'Prompt {i+1}: {p[:120]}...')

In [ ]:
# Generate with a FIXED seed for maximum consistency
SEED = 42
anchored_images = []

for i, p in enumerate(prompts):
    img = pipe.generate(
        prompt=p,
        negative_prompt=template.negative_prompt,
        seed=SEED,
        num_inference_steps=30,
    )
    img.save(OUTPUT_DIR / f'anchored_{i:02d}.png')
    anchored_images.append(img)

# Display as a grid
fig, axes = plt.subplots(1, len(anchored_images), figsize=(16, 4))
for ax, img, v in zip(axes, anchored_images, variations):
    ax.imshow(img)
    ax.set_title(v['action'][:30], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'anchored_grid.png', dpi=150)
plt.show()

---
## 2. Seed locking experiment

Compare outputs with a **fixed seed** vs **random seeds** to see the impact on consistency.

In [ ]:
from src.utils.seed_utils import generate_with_locked_seed, seed_range

LOCKED_SEED = 0
random_seeds = seed_range(42, 4)  # 4 diverse but deterministic seeds

base_prompt = 'sks character, anime explorer, {action}, masterpiece, best quality'
actions = ['on the moon', 'floating in space', 'planting a flag', 'inside a space station']
action_prompts = [base_prompt.format(action=a) for a in actions]

print('Generating with LOCKED seed...')
locked_images = generate_with_locked_seed(pipe, action_prompts, seed=LOCKED_SEED)

print('Generating with RANDOM seeds...')
random_images = [
    pipe.generate(prompt=p, seed=s)
    for p, s in zip(action_prompts, random_seeds)
]

In [ ]:
fig, axes = plt.subplots(2, len(actions), figsize=(16, 8))
for col, (l_img, r_img, action) in enumerate(zip(locked_images, random_images, actions)):
    axes[0, col].imshow(l_img)
    axes[0, col].set_title(f'Locked (seed={LOCKED_SEED})\n{action[:20]}', fontsize=7)
    axes[0, col].axis('off')
    axes[1, col].imshow(r_img)
    axes[1, col].set_title(f'Random (seed={random_seeds[col]})\n{action[:20]}', fontsize=7)
    axes[1, col].axis('off')

plt.suptitle('Locked seed (top) vs Random seeds (bottom)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'seed_comparison.png', dpi=150)
plt.show()

---
## 3. CLIP similarity scoring

Use CLIP embeddings to **automatically score** how consistent each output is
with a reference image.

In [ ]:
from src.utils.clip_blip_scoring import CLIPScorer

# TODO: Fill in src/utils/clip_blip_scoring.py before running.
clip = CLIPScorer(device='cuda')

In [ ]:
# Score all locked-seed images against the first one as reference
reference = locked_images[0]
scores = clip.batch_image_similarity(reference, locked_images)

print('CLIP image-image similarity scores (reference = first image):')
for i, (score, action) in enumerate(zip(scores, actions)):
    print(f'  {i}: {action:35s}  similarity = {score:.4f}')

In [ ]:
# Score against a text description (image-text similarity)
reference_text = 'an anime character in a space explorer suit, masterpiece'

print(f'Reference text: "{reference_text}"')
print('Image-text similarity scores:')
for i, (img, action) in enumerate(zip(locked_images, actions)):
    score = clip.image_text_similarity(img, reference_text)
    print(f'  {i}: {action:35s}  similarity = {score:.4f}')

---
## 4. ConsistencyScorer: multi-signal ranking

The `ConsistencyScorer` combines CLIP similarity and optional pixel-level metrics
to give a unified consistency score.  Use it to automatically filter a large batch
of generations.

In [ ]:
from src.utils.consistency_scoring import ConsistencyScorer

scorer = ConsistencyScorer(use_clip=True, use_pixel_mse=False, clip_weight=1.0)

# Generate a larger batch with random seeds
random_seeds_large = seed_range(0, 12)
PROMPT = template.build(action='standing on a mountain peak', setting='golden hour')

batch = [pipe.generate(PROMPT, seed=s) for s in random_seeds_large]

# Use the first locked-seed image as our reference
ranked = scorer.rank_by_consistency(reference=anchored_images[0], generated_images=batch)

print(f'Top 3 most consistent outputs (out of {len(batch)}):')
for rank, (score, img) in enumerate(ranked[:3], start=1):
    print(f'  Rank {rank}: score = {score:.4f}')
    img.save(OUTPUT_DIR / f'top_{rank}.png')
    display(img)

---
## 5. Latent space exploration

Encode a reference image to its VAE latent and add controlled noise to
generate near-reference variations.

In [ ]:
from src.utils.latent_utils import (
    add_noise_to_latent,
    decode_latent_to_image,
    encode_image_to_latent,
)

# TODO: Fill in src/utils/latent_utils.py before running.
# Requires access to the VAE from the pipeline (pipe._pipe.vae)

if pipe._pipe is not None:  # Only if pipeline is loaded
    vae = pipe._pipe.vae
    reference = anchored_images[0]

    latent = encode_image_to_latent(reference, vae)
    print(f'Latent shape: {latent.shape}')  # Expected: (1, 4, 64, 64) for 512px

    # Add small noise and decode — should look similar to reference
    for noise_lvl in [0.05, 0.1, 0.2, 0.4]:
        noised = add_noise_to_latent(latent, noise_level=noise_lvl, seed=42)
        decoded = decode_latent_to_image(noised / vae.config.scaling_factor, vae)
        decoded.save(OUTPUT_DIR / f'noised_latent_{noise_lvl:.2f}.png')
        print(f'noise_level={noise_lvl}')
        display(decoded)
else:
    print('Pipeline not loaded (stubs not implemented yet). Skipping latent exploration.')

---
## Summary

You've explored four complementary consistency approaches:

1. **Prompt anchoring** — cheapest, no compute overhead; most effective for style/character words
2. **Seed locking** — free, but consistency degrades with very different prompts
3. **CLIP scoring** — lets you automatically select the most consistent outputs from a large batch
4. **Latent noise** — allows near-reference variations with controllable divergence

For stronger subject consistency, combine these with **IP-Adapter** or **LoRA** (Notebook 02).